In [1]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer
from googletrans import Translator
import asyncio

In [2]:
class OpenDataSearch:
    def __init__(self, **kwargs):
        self.client = kwargs.get('client')
        self.encoder = kwargs.get('encoder')

        self.catalogs = {} # Armazena todos os catalogos pesquisados durante a sessão

    async def open_data_search(self, query: str) -> list:
        
        '''
        Name: open_data_search
        Description: Searches for datasets based on keywords.

        Arguments: 

        {
        "thought": "Brief reasoning about what to do next",
        "tool_name": "open_data_search",
        "parameters": {
            "query": "value"
            }
        }

        Return: 

            {id :
                {"score:": score,
                "id": id,
                "Titulo: ": title),
                "Nome: ": nome,
                "Descrição: ": descricao,
                }
            }
        '''


        translator = Translator()

        task = "Você é um motor de busca, devolva dados mais relevantes com base na busca"

        query_pt = f"Instrução: {task} Query: {query}"
        query_en = await translator.translate(query_pt, src='pt', dest='en')

        query = query_en.text

        hits = self.client.query_points(
            collection_name="Catalogo_metadados",
            query=self.encoder.encode(query).tolist(),
            limit=5,
        ).points

        catalogs = {} # catalogo temporario retornado, respectivo a cada query individual

        for hit in hits:
            id = hit.payload.get('id')
            if id not in catalogs.keys(): # garante que não há catalogos repitidos
                catalogs.update({ id :
                    {"score:": hit.score,
                    "id": id,
                    "Titulo: ": hit.payload.get('title'),
                    "Nome: ": hit.payload.get('nome'),
                    "Descrição: ": hit.payload.get('descricao'),
                    #"Nome organização": hit.payload.get('nomeOrganizacao'),
                    #"catalogacao": hit.payload.get('catalogacao'),
                    #"ultimaAtualizacaoDados'": hit.payload.get('ultimaAtualizacaoDados'), 
                    }
                })
        self.catalogs.update(catalogs) # Atualiza catalogos

        return catalogs
    
    def consult_catalogs(self, id: str) -> dict:
        '''
        tool_name = "consult_catalogs"
        
        '''
        return self.catalogs.get(id)


    def download_data(self):
        pass

    def clear_catalogs(self):
        self.catalogs.clear()

In [3]:
model_name="Qwen/Qwen3-Embedding-0.6B"
device='cpu'

In [4]:
encoder = SentenceTransformer(model_name, device=device)

In [5]:
client = QdrantClient(path="../metadados/VectorStore") # Carrega vectorstore em disco
open = OpenDataSearch(encoder=encoder, device=device, model_name=model_name, client=client)

In [6]:
query="infecção hospitalares"
await open.open_data_search(query)

{'0364f1c3-6198-4a17-982e-1d325ca8d8fd': {'score:': 0.706174920183519,
  'id': '0364f1c3-6198-4a17-982e-1d325ca8d8fd',
  'Titulo: ': '04. Infecção Hospitalar',
  'Nome: ': '04-infeccao-hospitalar',
  'Descrição: ': 'Taxa de infecção hospitalar.'},
 'e0561008-378a-4d0a-8ea7-fb37f0e2697f': {'score:': 0.6101069672041878,
  'id': 'e0561008-378a-4d0a-8ea7-fb37f0e2697f',
  'Titulo: ': 'Infecção Hospitalar',
  'Nome: ': 'infeccao-hospitalar',
  'Descrição: ': 'Quantidade de pacientes com Infecção Hospitalar'},
 '16c4b9c0-fbf5-49be-87b3-e91d00ace9b6': {'score:': 0.5993121379693481,
  'id': '16c4b9c0-fbf5-49be-87b3-e91d00ace9b6',
  'Titulo: ': '9. Taxa de Infecção Relacionada à Assistência em Saúde',
  'Nome: ': '9-taxa-de-infeccao-relacionada-a-assistencia-em-saude',
  'Descrição: ': 'Número de Infecção Relacionada à Assistência em Saúde do hospital no mês/saídas no mês X 100'},
 '9951c413-0274-4f94-9fb5-b883f4833228': {'score:': 0.5817487129996652,
  'id': '9951c413-0274-4f94-9fb5-b883f483322